In [1]:
!pip install -q transformers sentence-transformers datasets accelerate
!pip install -q gradio
!pip install -q faiss-gpu-cu12
!pip install -q pyarrow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 14.2 MB/s eta 0:00:00


In [2]:
# @title 1. Install Dependencies

import os
import shutil
import sys
import warnings

# Suppress minor warnings for cleaner output
warnings.filterwarnings('ignore')

print("✅ Dependencies installed successfully.")

✅ Dependencies installed successfully.


In [3]:
# @title 2. Configuration & Constants
import torch

class Config:
    # --- Paths (Google Drive & Local) ---
    DRIVE_MOUNT_PATH = "/content/drive"

    # Input Paths (Drive)
    BASE_DRIVE_PATH = "/content/drive/MyDrive/RAGSearch"
    PRODUCTS_PARQUET_PATH = f"{BASE_DRIVE_PATH}/preprocessed_products.parquet"
    IMAGES_DRIVE_PATH = f"{BASE_DRIVE_PATH}/balanced_images"
    TEXT_MODEL_CHECKPOINT = f"{BASE_DRIVE_PATH}/retrieval_system_full.pth"

    # Local Paths (Colab Runtime)
    LOCAL_IMAGE_DIR = "/content/local_images"

    # --- Model Names ---
    # Image Model (Vision Encoder)
    IMAGE_MODEL_ID = "google/siglip2-so400m-patch16-naflex"

    # Text Model Base (Must match the one used for training)
    TEXT_BASE_MODEL_ID = "HooshvareLab/distilbert-fa-zwnj-base"

    # --- Architecture Params (Matches PDF) ---
    MAX_LENGTH = 64
    PROJECTION_DIM = 128
    DROPOUT = 0.1

    # --- Indexing & Search ---
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 32
    TOP_K_RETRIEVAL = 40  # Retrieve 40 items total
    PAGE_SIZE = 10        # Show 10 per page

    # --- Seed ---
    SEED = 42

print(f"✅ Configuration loaded. Device: {Config.DEVICE}")

✅ Configuration loaded. Device: cuda


In [4]:
# @title 3. Data Preparation
from google.colab import drive
import pandas as pd
from tqdm.auto import tqdm
import glob

# 1. Mount Drive
if not os.path.exists(Config.DRIVE_MOUNT_PATH):
    print("Mounting Google Drive...")
    drive.mount(Config.DRIVE_MOUNT_PATH)

# 2. Load Product Metadata
print(f"Loading metadata from: {Config.PRODUCTS_PARQUET_PATH}")
df_products = pd.read_parquet(Config.PRODUCTS_PARQUET_PATH)
print(f"Total products in metadata: {len(df_products)}")

# 3. Copy Images to Local Disk
if not os.path.exists(Config.LOCAL_IMAGE_DIR):
    os.makedirs(Config.LOCAL_IMAGE_DIR)

    print(f"Copying images from {Config.IMAGES_DRIVE_PATH} to {Config.LOCAL_IMAGE_DIR}...")
    print("This may take a minute, but it significantly speeds up indexing...")

    # Using shell cp for speed
    !cp -r "{Config.IMAGES_DRIVE_PATH}/." "{Config.LOCAL_IMAGE_DIR}/"

    print("✅ Images copied successfully.")
else:
    print("ℹ️ Local image directory already exists. Skipping copy.")

# 4. Create Mappings
# Verify available images
available_images = set(os.listdir(Config.LOCAL_IMAGE_DIR))
print(f"Total local images available: {len(available_images)}")

# Filter dataframe for image search (only products with images)
# Assumes image filename is random_key.jpg
df_products['image_filename'] = df_products['random_key'] + ".jpg"
df_products['has_image'] = df_products['image_filename'].apply(lambda x: x in available_images)

# Split for different use cases
df_text_search = df_products.copy() # Text search uses ALL products
df_image_search = df_products[df_products['has_image']].copy() # Image search uses only available images

# Reset index for easier mapping
df_text_search = df_text_search.reset_index(drop=True)
df_image_search = df_image_search.reset_index(drop=True)

print(f"🔹 Products eligible for Text Search: {len(df_text_search)}")
print(f"🔹 Products eligible for Image Search: {len(df_image_search)}")

Mounting Google Drive...
Mounted at /content/drive
Loading metadata from: /content/drive/MyDrive/RAGSearch/preprocessed_products.parquet
Total products in metadata: 1022297
Copying images from /content/drive/MyDrive/RAGSearch/balanced_images to /content/local_images...
This may take a minute, but it significantly speeds up indexing...
✅ Images copied successfully.
Total local images available: 9151
🔹 Products eligible for Text Search: 1022297
🔹 Products eligible for Image Search: 9151


In [5]:
# @title 4. Text Model Architecture (Reconstructed)
import torch.nn as nn
from transformers import AutoModel, AutoConfig

class EncoderWithHeads(nn.Module):
    def __init__(self, base_model_path, num_classes, embedding_dim):
        super().__init__()
        # Load Base Model
        self.bert = AutoModel.from_pretrained(base_model_path)
        hidden_size = self.bert.config.hidden_size

        # Contrastive Head (Projection)
        self.contrastive_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(hidden_size, embedding_dim)
        )

        # Classification Head (Used during training, kept for state_dict compatibility)
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        # Base Model Forward
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Pooling: Use CLS token (index 0)
        cls_token = outputs.last_hidden_state[:, 0, :]

        # Heads
        emb = self.contrastive_head(cls_token)
        # Normalize embedding for Cosine Similarity
        emb = nn.functional.normalize(emb, p=2, dim=1)

        # We don't strictly need logits for inference, but the class structure expects it
        logits = self.classification_head(cls_token)

        return emb, logits

class RetrievalSystem(nn.Module):
    def __init__(self, base_model_path, num_classes):
        super().__init__()
        # Two DISTINCT encoders
        self.query_encoder = EncoderWithHeads(base_model_path, num_classes, Config.PROJECTION_DIM)
        self.product_encoder = EncoderWithHeads(base_model_path, num_classes, Config.PROJECTION_DIM)

    def forward(self, batch):
        # Placeholder forward for training structure, not used in inference
        pass

print("✅ Text Model Architecture defined.")

✅ Text Model Architecture defined.


In [7]:
# @title 5. Load Models (Fixed)
import torch
from transformers import AutoProcessor, AutoTokenizer

# --- 1. Load Image Model (SigLIP) ---
print("Loading Image Model (SigLIP)...")
image_processor = AutoProcessor.from_pretrained(Config.IMAGE_MODEL_ID)
image_model = AutoModel.from_pretrained(Config.IMAGE_MODEL_ID, device_map="auto").eval()
print("✅ Image Model loaded.")

# --- 2. Load Text Model ---
print("Loading Text Model (Custom DistilBERT)...")
text_tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_BASE_MODEL_ID)

# CHECKPOINT LOADING LOGIC
if os.path.exists(Config.TEXT_MODEL_CHECKPOINT):
    # 1. Load state_dict to CPU first to inspect it
    print(f"Loading weights from: {Config.TEXT_MODEL_CHECKPOINT}")
    state_dict = torch.load(Config.TEXT_MODEL_CHECKPOINT, map_location='cpu')

    # 2. Smart Detection of num_classes (FIX FOR THE BUG)
    # The error happened because current data has 687 classes, but checkpoint has 674.
    # We must initialize the architecture to match the CHECKPOINT, not the current dataframe.
    # We inspect the final layer of the classification head.
    key_to_inspect = 'query_encoder.classification_head.3.weight'

    if key_to_inspect in state_dict:
        checkpoint_classes = state_dict[key_to_inspect].shape[0]
        print(f"ℹ️ Detected {checkpoint_classes} classes in checkpoint weights.")
        num_classes = checkpoint_classes
    else:
        print("⚠️ Could not detect classes from weights. Defaulting to dataframe count.")
        num_classes = df_products['category_id'].nunique()

    # 3. Initialize Model with the CORRECT number of classes
    text_model = RetrievalSystem(Config.TEXT_BASE_MODEL_ID, num_classes)

    # 4. Load weights
    try:
        text_model.load_state_dict(state_dict)
        print("✅ Text Model loaded (Strict Mode).")
    except RuntimeError as e:
        print(f"⚠️ Strict loading failed. Attempting to filter classification head...")
        # If there is still an issue, we remove the classification head keys entirely
        # This is safe because we only need the ENCODERS for retrieval.
        model_dict = text_model.state_dict()

        # Filter out keys that don't match in shape (specifically the classification head)
        filtered_state_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}

        text_model.load_state_dict(filtered_state_dict, strict=False)
        print("✅ Text Model loaded (Encoder Only - Classification Head filtered).")

    text_model.to(Config.DEVICE)
    text_model.eval()

else:
    raise FileNotFoundError(f"Checkpoint not found at {Config.TEXT_MODEL_CHECKPOINT}")

Loading Image Model (SigLIP)...
✅ Image Model loaded.
Loading Text Model (Custom DistilBERT)...
Loading weights from: /content/drive/MyDrive/RAGSearch/retrieval_system_full.pth
ℹ️ Detected 674 classes in checkpoint weights.
✅ Text Model loaded (Strict Mode).


In [8]:
# @title 6. Indexing Engine
import faiss
import numpy as np
from PIL import Image

# Initialize FAISS Indices (Inner Product = Cosine Similarity for normalized vectors)
# Config.PROJECTION_DIM is 128 for Text
# SigLIP dimension is usually 1152 for so400m
siglip_config = image_model.config
vision_hidden_dim = siglip_config.vision_config.hidden_size # likely 1152

# Resources for FAISS GPU
res = faiss.StandardGpuResources()

# --- Text Index Setup ---
text_index_cpu = faiss.IndexFlatIP(Config.PROJECTION_DIM)
text_index = faiss.index_cpu_to_gpu(res, 0, text_index_cpu)

# --- Image Index Setup ---
image_index_cpu = faiss.IndexFlatIP(vision_hidden_dim)
image_index = faiss.index_cpu_to_gpu(res, 0, image_index_cpu)

# --- Batch Indexing: TEXT ---
print("\n🚀 Starting Text Indexing (All Products)...")
text_data = df_text_search['cleaned_text'].tolist()
# Check for nulls and fill
text_data = [str(t) if t is not None else "" for t in text_data]

batch_size = Config.BATCH_SIZE
total_text = len(text_data)

for i in tqdm(range(0, total_text, batch_size), desc="Indexing Text"):
    batch_texts = text_data[i : i + batch_size]

    # Tokenize
    inputs = text_tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=Config.MAX_LENGTH,
        return_tensors="pt"
    ).to(Config.DEVICE)

    with torch.no_grad():
        # Use PRODUCT encoder for indexing database items
        emb, _ = text_model.product_encoder(inputs['input_ids'], inputs['attention_mask'])
        emb_np = emb.cpu().numpy().astype('float32')

    text_index.add(emb_np)

print(f"✅ Text Indexing Complete. Total vectors: {text_index.ntotal}")

# --- Batch Indexing: IMAGES ---
print("\n🚀 Starting Image Indexing (Local Images Only)...")
image_keys = df_image_search['random_key'].tolist()
total_images = len(image_keys)

for i in tqdm(range(0, total_images, batch_size), desc="Indexing Images"):
    batch_keys = image_keys[i : i + batch_size]
    batch_images = []
    valid_indices = []

    for idx, key in enumerate(batch_keys):
        img_path = os.path.join(Config.LOCAL_IMAGE_DIR, f"{key}.jpg")
        try:
            img = Image.open(img_path).convert("RGB")
            batch_images.append(img)
            valid_indices.append(idx) # Keep track if file read fails (though we filtered earlier)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

    if not batch_images:
        continue

    inputs = image_processor(images=batch_images, return_tensors="pt").to(Config.DEVICE)

    with torch.no_grad():
        # Get vision features
        image_embeddings = image_model.get_image_features(**inputs)
        # Normalize for Cosine Similarity (SigLIP features aren't strictly normalized by default output)
        image_embeddings = torch.nn.functional.normalize(image_embeddings, p=2, dim=1)
        emb_np = image_embeddings.cpu().numpy().astype('float32')

    image_index.add(emb_np)

print(f"✅ Image Indexing Complete. Total vectors: {image_index.ntotal}")


🚀 Starting Text Indexing (All Products)...


Indexing Text:   0%|          | 0/31947 [00:00<?, ?it/s]

✅ Text Indexing Complete. Total vectors: 1022297

🚀 Starting Image Indexing (Local Images Only)...


Indexing Images:   0%|          | 0/286 [00:00<?, ?it/s]

The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


✅ Image Indexing Complete. Total vectors: 9151


In [9]:
# @title 7. Search Logic Implementation

def search_text(query_text):
    """
    1. Encode query using Query Encoder.
    2. Search FAISS.
    3. Retrieve metadata.
    """
    if not query_text.strip():
        return []

    # Tokenize
    inputs = text_tokenizer(
        [query_text],
        padding=True,
        truncation=True,
        max_length=Config.MAX_LENGTH,
        return_tensors="pt"
    ).to(Config.DEVICE)

    # Encode with QUERY encoder
    with torch.no_grad():
        q_emb, _ = text_model.query_encoder(inputs['input_ids'], inputs['attention_mask'])
        q_emb_np = q_emb.cpu().numpy().astype('float32')

    # Search
    D, I = text_index.search(q_emb_np, Config.TOP_K_RETRIEVAL)

    # Format Results
    results = []
    indices = I[0]
    distances = D[0]

    for rank, idx in enumerate(indices):
        if idx == -1: continue # FAISS padding

        row = df_text_search.iloc[idx]

        # Determine Brand
        brand = row['brand_name'] if row['brand_id'] != -1 else "No Brand"

        item = {
            "rank": rank + 1,
            "score": float(distances[rank]),
            "persian_name": row['persian_name'],
            "category_name": row['category_name'],
            "brand_name": brand,
            # For text search we don't strictly require image, but if we have it locally, we can send path
            "image_path": os.path.join(Config.LOCAL_IMAGE_DIR, f"{row['random_key']}.jpg") if row['random_key'] + ".jpg" in available_images else None
        }
        results.append(item)

    return results

def search_image(query_image):
    """
    1. Encode uploaded image using SigLIP.
    2. Search FAISS.
    3. Retrieve metadata.
    """
    if query_image is None:
        return []

    # Encode
    inputs = image_processor(images=[query_image], return_tensors="pt").to(Config.DEVICE)

    with torch.no_grad():
        img_emb = image_model.get_image_features(**inputs)
        img_emb = torch.nn.functional.normalize(img_emb, p=2, dim=1)
        q_emb_np = img_emb.cpu().numpy().astype('float32')

    # Search
    D, I = image_index.search(q_emb_np, Config.TOP_K_RETRIEVAL)

    # Format Results
    results = []
    indices = I[0]
    distances = D[0]

    for rank, idx in enumerate(indices):
        if idx == -1: continue

        row = df_image_search.iloc[idx] # Use image dataframe

        brand = row['brand_name'] if row['brand_id'] != -1 else "No Brand"
        img_path = os.path.join(Config.LOCAL_IMAGE_DIR, f"{row['random_key']}.jpg")

        item = {
            "rank": rank + 1,
            "score": float(distances[rank]),
            "persian_name": row['persian_name'],
            "category_name": row['category_name'],
            "brand_name": brand,
            "image_path": img_path
        }
        results.append(item)

    return results

In [10]:
# @title 8. Gradio GUI
import gradio as gr
import math

# --- Pagination Logic Helpers ---

def get_page_data(results, page_num):
    """Slice results based on page number"""
    start = (page_num - 1) * Config.PAGE_SIZE
    end = start + Config.PAGE_SIZE
    return results[start:end]

def render_text_results(results_page):
    """HTML generator for Text Search Results"""
    html = "<div style='display: flex; flex-direction: column; gap: 15px;'>"
    for item in results_page:
        html += f"""
        <div style="border: 1px solid #ddd; padding: 10px; border-radius: 8px; background-color: #f9f9f9;">
            <h3 style="margin: 0; color: #333;">{item['persian_name']}</h3>
            <p style="margin: 5px 0;"><b>Category:</b> {item['category_name']} | <b>Brand:</b> {item['brand_name']}</p>
            <p style="font-size: 0.8em; color: #666;">Similarity Score: {item['score']:.4f}</p>
        </div>
        """
    html += "</div>"
    if not results_page:
        html = "<p>No results found.</p>"
    return html

def render_image_results(results_page):
    """
    Format data for Gradio Gallery.
    Returns list of (image_path, caption).
    """
    gallery_data = []
    for item in results_page:
        caption = f"{item['persian_name']}\n{item['category_name']} | {item['brand_name']}"
        gallery_data.append((item['image_path'], caption))
    return gallery_data

# --- Event Handlers ---

# Text Search Handler
def on_text_search(query):
    full_results = search_text(query)
    current_page = 1
    total_pages = math.ceil(len(full_results) / Config.PAGE_SIZE)

    page_data = get_page_data(full_results, current_page)
    html_output = render_text_results(page_data)

    nav_info = f"Page {current_page} of {total_pages}"

    # Return: StoredResults, HTML, PageNum, TotalPages, NavLabel, PrevBtnState, NextBtnState
    return (
        full_results,
        html_output,
        current_page,
        total_pages,
        nav_info,
        gr.update(interactive=False), # Prev
        gr.update(interactive=(current_page < total_pages)) # Next
    )

# Image Search Handler
def on_image_search(image):
    full_results = search_image(image)
    current_page = 1
    total_pages = math.ceil(len(full_results) / Config.PAGE_SIZE)

    page_data = get_page_data(full_results, current_page)
    gallery_output = render_image_results(page_data)

    nav_info = f"Page {current_page} of {total_pages}"

    return (
        full_results,
        gallery_output,
        current_page,
        total_pages,
        nav_info,
        gr.update(interactive=False),
        gr.update(interactive=(current_page < total_pages))
    )

# Pagination Handler (Generic for both)
def on_page_change(action, current_results, current_page, total_pages, mode="text"):
    if not current_results:
        return gr.update(), current_page, gr.update(), gr.update(), gr.update()

    new_page = current_page
    if action == "next" and current_page < total_pages:
        new_page += 1
    elif action == "prev" and current_page > 1:
        new_page -= 1

    page_data = get_page_data(current_results, new_page)

    nav_info = f"Page {new_page} of {total_pages}"
    prev_state = gr.update(interactive=(new_page > 1))
    next_state = gr.update(interactive=(new_page < total_pages))

    if mode == "text":
        output_content = render_text_results(page_data)
    else:
        output_content = render_image_results(page_data)

    return output_content, new_page, nav_info, prev_state, next_state


# --- UI Construction ---

custom_css = """
.gradio-container { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
"""

with gr.Blocks(css=custom_css, title="AI Product Retrieval") as demo:
    gr.Markdown("# 🛍️ AI Vector Product Retrieval System")
    gr.Markdown("Dual-Encoder Architecture (DistilBERT + SigLIP)")

    with gr.Tabs():
        # --- TAB 1: Image 2 Image ---
        with gr.TabItem("🖼️ Image-to-Image Search"):
            with gr.Row():
                with gr.Column(scale=1):
                    img_input = gr.Image(type="pil", label="Upload Query Image")
                    img_search_btn = gr.Button("Search Similar Products", variant="primary")

                with gr.Column(scale=2):
                    # State
                    img_results_state = gr.State([])
                    img_page_state = gr.State(1)
                    img_total_pages_state = gr.State(1)

                    # Output
                    img_output = gr.Gallery(label="Results", columns=2, height=600, object_fit="contain")

                    # Pagination Controls
                    with gr.Row():
                        img_prev_btn = gr.Button("Previous", interactive=False)
                        img_page_info = gr.Label("Page 1 of 1", show_label=False)
                        img_next_btn = gr.Button("Next", interactive=False)

            # Events
            img_search_btn.click(
                on_image_search,
                inputs=[img_input],
                outputs=[img_results_state, img_output, img_page_state, img_total_pages_state, img_page_info, img_prev_btn, img_next_btn]
            )

            img_prev_btn.click(
                lambda r, p, t: on_page_change("prev", r, p, t, "image"),
                inputs=[img_results_state, img_page_state, img_total_pages_state],
                outputs=[img_output, img_page_state, img_page_info, img_prev_btn, img_next_btn]
            )

            img_next_btn.click(
                lambda r, p, t: on_page_change("next", r, p, t, "image"),
                inputs=[img_results_state, img_page_state, img_total_pages_state],
                outputs=[img_output, img_page_state, img_page_info, img_prev_btn, img_next_btn]
            )

        # --- TAB 2: Text 2 Text ---
        with gr.TabItem("📝 Text-to-Text Search"):
            with gr.Row():
                with gr.Column(scale=1):
                    txt_input = gr.Textbox(label="Enter Query", placeholder="e.g., فرش ماشینی آبی")
                    txt_search_btn = gr.Button("Search Products", variant="primary")

                with gr.Column(scale=2):
                    # State
                    txt_results_state = gr.State([])
                    txt_page_state = gr.State(1)
                    txt_total_pages_state = gr.State(1)

                    # Output
                    txt_output = gr.HTML()

                    # Pagination Controls
                    with gr.Row():
                        txt_prev_btn = gr.Button("Previous", interactive=False)
                        txt_page_info = gr.Label("Page 1 of 1", show_label=False)
                        txt_next_btn = gr.Button("Next", interactive=False)

            # Events
            txt_search_btn.click(
                on_text_search,
                inputs=[txt_input],
                outputs=[txt_results_state, txt_output, txt_page_state, txt_total_pages_state, txt_page_info, txt_prev_btn, txt_next_btn]
            )

            txt_prev_btn.click(
                lambda r, p, t: on_page_change("prev", r, p, t, "text"),
                inputs=[txt_results_state, txt_page_state, txt_total_pages_state],
                outputs=[txt_output, txt_page_state, txt_page_info, txt_prev_btn, txt_next_btn]
            )

            txt_next_btn.click(
                lambda r, p, t: on_page_change("next", r, p, t, "text"),
                inputs=[txt_results_state, txt_page_state, txt_total_pages_state],
                outputs=[txt_output, txt_page_state, txt_page_info, txt_prev_btn, txt_next_btn]
            )

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://46debfb3a9d985143d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://46debfb3a9d985143d.gradio.live
